### 07b · Diagnóstico: os sinais separam os ativos ou é ruído?
####  **Fase 3 · Gold** | Entrada: `silver.telemetria_clean`
#### 
###  Para cada sinal, compara a **dispersão observada** entre os 50 dispositivos com a **dispersão esperada só por acaso** (erro padrão da amostragem).
#### 
####  `razao = desvio_observado / desvio_esperado`
####  - perto de 1: ruído, o sinal não separa ativos
####  - bem acima de 1 (> 2): há diferença real entre dispositivos
# 
####  Os sinais de comportamento usam o resíduo contra a média da frota na mesma hora do dia, para tirar o ciclo diário da temperatura.
#### """' Só entram no score os sinais com razão claramente acima de 1.

In [0]:
CATALOG = "project_lakehouse_telemetria_ele"
SILVER = f"{CATALOG}.silver.telemetria_clean"
 
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_diag AS
WITH b AS (
  SELECT device_id, HOUR(event_ts) AS h,
         COALESCE(CAST(temp_pico AS INT),0) AS f_pico,
         COALESCE(CAST(tensao_ausente AS INT),0) AS f_tens,
         COALESCE(CAST(corrente_sentinela AS INT),0) AS f_sent,
         CASE WHEN COALESCE(CAST(temp_pico AS INT),0)=0 AND COALESCE(CAST(corrente_sentinela AS INT),0)=0
              THEN temperatura_c END AS temp_ok,
         CASE WHEN COALESCE(CAST(temp_pico AS INT),0)=0 AND COALESCE(CAST(corrente_sentinela AS INT),0)=0
              THEN corrente_a END AS corr_ok,
         CASE WHEN COALESCE(CAST(temp_pico AS INT),0)=0 AND COALESCE(CAST(corrente_sentinela AS INT),0)=0
              THEN tensao_v END AS tens_ok
  FROM {SILVER}
),
r AS (
  SELECT *, temp_ok - AVG(temp_ok) OVER (PARTITION BY h) AS temp_res FROM b
)
SELECT device_id, metrica, COUNT(valor) AS n, AVG(valor) AS media, STDDEV(valor) AS sd
FROM (
  SELECT device_id, 'taxa_picos (= alarme)' AS metrica, CAST(f_pico AS DOUBLE) AS valor FROM r
  UNION ALL SELECT device_id, 'taxa_tensao_ausente', CAST(f_tens AS DOUBLE) FROM r
  UNION ALL SELECT device_id, 'taxa_corrente_sentinela', CAST(f_sent AS DOUBLE) FROM r
  UNION ALL SELECT device_id, 'temp_residual_hora', temp_res FROM r
  UNION ALL SELECT device_id, 'corrente_media', corr_ok FROM r
  UNION ALL SELECT device_id, 'tensao_media', tens_ok FROM r
)
GROUP BY device_id, metrica
""")

In [0]:
display(spark.sql("""
SELECT metrica,
       COUNT(*) AS dispositivos,
       ROUND(AVG(media),4) AS media_geral,
       ROUND(STDDEV(media),5) AS desvio_observado,
       ROUND(SQRT(AVG(sd*sd/n)),5) AS desvio_esperado_acaso,
       ROUND(STDDEV(media)/SQRT(AVG(sd*sd/n)),2) AS razao,
       CASE WHEN STDDEV(media)/SQRT(AVG(sd*sd/n)) > 2 THEN 'sinal real'
            WHEN STDDEV(media)/SQRT(AVG(sd*sd/n)) > 1.5 THEN 'indício, checar'
            ELSE 'ruído' END AS leitura
FROM v_diag GROUP BY metrica ORDER BY razao DESC
"""))

### Como usar o resultado
### Sinais marcados `ruído` não entram no score (ou entram com peso baixo e declarado como tal no README).
### Se **nenhum** sinal for real, o ranking de risco não tem base estatística: o entregável honesto é dizer isso e mostrar o intervalo de confiança dos ativos do topo.
### Se algum sinal de comportamento for real (por exemplo `temp_residual_hora`), ele vira o eixo principal do score no notebook 09.
### Premissa da razão: leituras independentes. Com autocorrelação no tempo, o desvio esperado real é um pouco maior, então a razão aqui é levemente otimista.